In [66]:

import pandas as pd
#print(pd.__version__)

from dotenv import load_dotenv
import os
from supabase import create_client

load_dotenv()

#print(os.getenv("SUPABASE_URL"))

supabase = create_client(os.getenv("SUPABASE_URL"),
                           os.getenv("SUPABASE_KEY"))

def get_data(table_name):
    data = []
    page_size = 1000
    page = 0
    while True:
        response = supabase.table(table_name).select("*").range(page * page_size, (page + 1) * page_size - 1).execute()
        data.extend(response.data)
        if len(response.data) < page_size:
            break
        page += 1

    return pd.DataFrame(data)

# 1. Laadime müügiandmed (df_sales)
df_sales = get_data("sales")

print(df_sales.head())

# 2. Laadime kliendiandmed (df_customers) — SELLE LISASTATI JUURDE!
df_customers = get_data("customers")

print(df_customers.head())


print(sales_df.head())
print(sales_df.shape)

# 3. Liidame kaks tabelit kokku (merge)

# Kui df_customers ei ole tühi, teeme merge'i. Muidu kasutame otse df_sales tabelit.
if not df_customers.empty:
    right_key = 'customer_id' if 'customer_id' in df_customers.columns else 'id'
    df = pd.merge(df_sales, df_customers, left_on='customer_id', right_on=right_key, how='left')
else:
    print("Märkus: eraldiseisvat klienditabelit ei leitud. Kasutame müügitabelit df_sales.")
    df = df_sales.copy()

print("\nLõpliku tabeli esimesed 5 rida (head):")
print(df.head())
print("Mõõtmed (shape):", df.shape)
print("\nVeergude andmetüübid (dtypes):\n", df.dtypes)

   id  sale_id        invoice_id   sale_date  customer_id  product_id  \
0   1        1  INV-202301-00001  2023-01-10       2588.0        1274   
1   2        2  INV-202301-00002  2023-01-16       4338.0        1207   
2   3        3  INV-202301-00003  2023-01-05       4673.0        1264   
3   4        4  INV-202301-00004  2023-01-02       4677.0        1341   
4   5        5  INV-202301-00005  2023-01-13       2390.0        1284   

   quantity  unit_price  total_price channel store_location payment_method  
0         2      234.79       469.58    pood        Tallinn          kaart  
1         2      241.13       482.26    pood          Pärnu      järelmaks  
2         1      258.46       221.19    pood          Pärnu      järelmaks  
3         3       45.21       135.63    pood          Tartu       sularaha  
4         1       99.57        99.57    pood          Tartu          kaart  
Empty DataFrame
Columns: []
Index: []
   id  sale_id        invoice_id   sale_date  customer_id  pr

Roll C – RFM Analysis

Eesmärk: arvutada klientidele Recency, Frequency ja Monetary näitajad ning moodustada RFM segmendid.

In [67]:
# 1. Leia TOP 5 toodet kogukäibe järgi

# 1. Leia TOP 5 toodet kogukäibe järgi
top_products = sales_df.groupby('product_id')['total_price'].sum().nlargest(5)
print(top_products)

product_id
1031    27347.04
1108    23376.15
1027    22188.80
1006    22039.98
1049    21309.96
Name: total_price, dtype: float64


In [68]:
# 2. Leia klient suurima kogukulutusega
top_customer = sales_df.groupby('customer_id')['total_price'].sum().idxmax()
print("Top customer:", top_customer)

Top customer: 3618.0


In [69]:
import pandas as pd

# 1. Veendume, et kuupäev on õiges vormingus (Roll B puhastussamm)
df['sale_date'] = pd.to_datetime(df['sale_date'])

# 2. Määrame viitekuupäeva vastavalt juhendile
today = pd.to_datetime('2025-02-28')

# 3. Arvutame iga kliendi viimase ostu ja päevade arvu (Recency)
recency = df.groupby('customer_id')['sale_date'].max().reset_index()
recency.columns = ['customer_id', 'last_purchase']
recency['recency_days'] = (today - recency['last_purchase']).dt.days

print(recency.head())

# 2. ETAPP: Frequency (ostude koguarv kliendi kohta)
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

print(frequency.head())

# SAMM 3: Frequency (ostude arv)
frequency = df.groupby('customer_id')['sale_id'].count().reset_index()
frequency.columns = ['customer_id', 'frequency']

print("Frequency (esimesed 5 rida):")
print(frequency.head())

#SAMM 4: Monetary (kogukulutus kliendi kohta)
monetary = df.groupby('customer_id')['total_price'].sum().reset_index()
monetary.columns = ['customer_id', 'monetary']

print("Monetary (esimesed 5 rida):")
print(monetary.head())

import pandas as pd

# 1. LIITMINE: Paneme recency, frequency ja monetary ühte rfm tabelisse kokku
rfm = recency[['customer_id', 'recency_days']].merge(frequency, on='customer_id').merge(monetary, on='customer_id')

# 2. SKOORID (1-5): Arvutame kvintiilide alusel skoorid (täpselt 5 silti iga näitaja jaoks)
rfm['R_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# 3. KOGUSKOOR: Liidame R + F + M skoorid kokku (vahemik 3 kuni 15)
rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# 4. SEGMENTIDE LOOMINE: Omistame ärisildid
def segment_customer(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(segment_customer)

# 5. ROLL C AMETLIK KVALITEEDIKONTROLL JA VÄLJUND
print("=== ROLL C LÕPPLIK RFM SEGMENTIDE KOKKUVÕTE ===")
print(rfm['Segment'].value_counts())
print("\nSegmentide osakaalud (%):\n", (rfm['Segment'].value_counts(normalize=True) * 100).round(1))
print("\nEsimesed 5 klienti koos segmentidega:")
print(rfm.head())

   customer_id last_purchase  recency_days
0       2001.0    2024-11-29            91
1       2004.0    2024-12-19            71
2       2005.0    2024-10-03           148
3       2006.0    2023-11-09           477
4       2007.0    2025-01-30            29
   customer_id  frequency
0       2001.0          2
1       2004.0          2
2       2005.0          4
3       2006.0          1
4       2007.0          1
Frequency (esimesed 5 rida):
   customer_id  frequency
0       2001.0          2
1       2004.0          2
2       2005.0          4
3       2006.0          1
4       2007.0          1
Monetary (esimesed 5 rida):
   customer_id  monetary
0       2001.0    203.92
1       2004.0   1198.56
2       2005.0    959.60
3       2006.0    327.06
4       2007.0    318.63
=== ROLL C LÕPPLIK RFM SEGMENTIDE KOKKUVÕTE ===
Segment
Potential        770
Loyal            691
At Risk          535
VIP Champions    446
Lost             109
Name: count, dtype: int64

Segmentide osakaalud (%):
 Segment
